In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [4]:
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [6]:
%pip install massive


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from backend.accounts import Account

In [8]:
account = Account.get("Mohan")
account.reset()
account

Account(name='mohan', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [9]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "mohan", "balance": 9120.4444, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 293.1852, "timestamp": "2026-07-24 16:56:54", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-07-24 16:56:54", 9998.2444]], "total_portfolio_value": 9998.2444, "total_profit_loss": -1.7556000000004133}'

In [10]:
account.report()

'{"name": "mohan", "balance": 9120.4444, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 293.1852, "timestamp": "2026-07-24 16:56:54", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-07-24 16:56:54", 9998.2444], ["2026-07-24 16:57:24", 9998.3044]], "total_portfolio_value": 9998.3044, "total_profit_loss": -1.6955999999991036}'

In [12]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 293.1852,
  'timestamp': '2026-07-24 16:56:54',
  'rationale': 'Because this bookstore website looks promising'}]

In [15]:
import os
project_dir = os.path.abspath(os.path.join(os.getcwd()))
params = {"command": "uv", "args": ["run", "-m", "backend.accounts_server"], "cwd": project_dir}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [16]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='buy_s

In [17]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Mohan and my account is under the name raj. What's my balance and my holdings?"
model = "gpt-4o-mini"

In [18]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Your current balance is **$10,000**. 

You don't have any holdings in your account at the moment. If you need assistance with anything else, feel free to ask!